# Outline

- Preparing Workspace

Importing packages, defining file paths, running user defined functions, setting API key, ...

- Preparing Imports

This section imports the "Census Configuration File.xlsx" and sets the user defined inputs to objects that are used throughout the script

- Importing
- Processing
- Exporting

***

## Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Census')
    path_config  = os.path.join(path_code, 'aa_config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Census')
    path_config  = os.path.join(path_code, 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

## Preparing Imports

***

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying Census data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'    ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'       ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'   ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'      ]['Input'].values[0]

# View
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# Import about table
df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
MOE_thresh = df_about['MOE Threshold'].values[0]
print(folder)
print('MOE threshold: ' + str(MOE_thresh) + '%')

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'Step 01 - Queries', 'Step 01 - Prepare API Request Inputs.py')).read())

# View result
df_vars.head(3)

***

## Importing

***

In [ ]:
# Execute script to import Census Bureau Data
exec(open(os.path.join(path_code, 'Step 01 - Queries', 'Step 02 - Run API Queries.py')).read())

In [ ]:
# View result
print(df_census_raw.shape)
print(df_census_raw.Year.unique())
pd.set_option('display.max_columns', None)
display(df_census_raw)

***

## Processing

***

In [ ]:
## Make copy of raw data
df_census = df_census_raw.copy()

In [ ]:

# Replace weird missing values with np.nan
# Melt data from wide to long
# Convert imported values to numeric
# Merge cleam label field, variable mapping, race/ethnicity, and sorting field
# Remove unneeded columns
# Manually check column names and clean as needed
# Adjust dollars for inflation, if needed
if sample_type in ['ACS', 'SUBJECT']:
    df_census = acs_processing_1(df_census, df_vars, indicator_name, geography, year_end, path_main, path_git)
    display(df_census.head(3))

# Reorganize margin of error fields
# Create "Categorical" race/ethnicity field for sorting
# Sort by geography, variable mapping, and race/ethnicity
# sort and then remove categorical field
if sample_type in ['ACS', 'SUBJECT']:
    df_census = acs_processing_2(df_census, geography, margin_of_error)
    display(df_census.head(3))

# Final processing step for ACS data
# Link various FIPS codes
# Roll up population/households/SE's to the desired geography and variable groupings
# Calculate percentages by geography, race/ethnicity, and variables
if sample_type in ['ACS', 'SUBJECT']:
    if geography == 'Tracts':
        df_tracts1, df_tracts2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars, df_fips)
        display(df_tracts1.head(3), df_tracts2.head(3))
    if geography == 'Counties':
        df_counties1, df_counties2, df_mpo1, df_mpo2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars, df_fips)
        display(df_mpo1.head(3), df_mpo2.head(3))
    if geography == 'MSA':
        df_msa1, df_msa2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars)
        display(df_msa1.head(3), df_msa2.head(3))
        

In [ ]:

# Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
# Convert weighted column to integer, convert value fields to string to use as merge field
# Reshape data dictionary of values/descriptions and reorganize columns
# Merge meaningful value descriptions onto imported data
if sample_type in ['PUMS', 'FOODSEC']:
    df_census, groups = pums_processing_1(df_census, df_vars, sample_type, weight)
    print('Groups: ' + ', '.join(groups))
    display(df_census.head(3))

# Remove rows with missing values
# Only keep description mappings, remove the original PUMS values
if sample_type in ['PUMS', 'FOODSEC']:
    df_census, groups = pums_processing_2(df_census, groups, indicator_name, dict_fips, path_config0, path_git)
    print('Groups: ' + ', '.join(groups))
    display(df_census.head(3))

# Roll up using suggested weight field
# Roll up to PUMA, counties, MSA, and MPO
if sample_type == 'PUMS':
    df_puma, df_counties, df_msa, df_mpo = pums_processing_3(df_census, indicator_name, weight, margin_of_error, MOE_thresh, percentages, groups)
    display(df_puma.head(3), df_counties.head(3), df_msa.head(3), df_mpo.head(3))

# Roll up using suggested weight field
# Roll up to counties and MPO
if sample_type == 'FOODSEC':
    df_counties, df_mpo, groups = food_processing_3(df_census, weight, percentages, groups)
    display(df_counties.head(3), df_mpo.head(3))


In [ ]:

# Final organization of tables for cleanliness
# Renaming columns, subsetting to only desired columns, ...

print('Final Results: ')
print('')

if geography == 'Tracts':
    df_tracts1 = rename_census(df_tracts1        = df_tracts1
                               , geography       = geography
                               , indicator_name  = indicator_name
                               , margin_of_error = margin_of_error
                               , sample_type     = sample_type)
    display(df_tracts1.head(3))
if geography == 'Counties':
    if sample_type in ['ACS', 'SUBJECT']:
        df_counties1, df_mpo1 = rename_census(df_counties1      = df_counties1
                                              , df_mpo1         = df_mpo1
                                              , geography       = geography
                                              , indicator_name  = indicator_name
                                              , margin_of_error = margin_of_error
                                              , sample_type     = sample_type)
        display(df_counties1.head(3), df_mpo1.head(3))
    if sample_type == 'FOODSEC':
        df_counties, df_mpo = rename_census(df_counties         = df_counties
                                              , df_mpo          = df_mpo
                                              , geography       = geography
                                              , indicator_name  = indicator_name
                                              , margin_of_error = margin_of_error
                                              , sample_type     = sample_type
                                              , table_type      = table_type
                                              , groups          = groups)
        display(df_counties.head(3), df_mpo.head(3))
if geography == 'MSA':
    df_msa1 = rename_census(df_msa1           = df_msa1
                            , geography       = geography
                            , indicator_name  = indicator_name
                            , margin_of_error = margin_of_error
                            , sample_type     = sample_type)
    display(df_msa1.head(3))
if geography == 'PUMA':
    df_puma, df_counties, df_msa, df_mpo = rename_census(df_puma           = df_puma
                                                         , df_counties     = df_counties
                                                         , df_msa          = df_msa
                                                         , df_mpo          = df_mpo
                                                         , geography       = geography
                                                         , indicator_name  = indicator_name
                                                         , margin_of_error = margin_of_error
                                                         , sample_type     = sample_type
                                                         , groups          = groups
                                                         , table_type      = table_type)
    display(df_puma.head(3), df_counties.head(3), df_msa.head(3), df_mpo.head(3))


***

## Exporting

***

In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )

if sample_type in ['ACS', 'SUBJECT']:
    if geography == 'Tracts':
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
        with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Tracts '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_tracts1.to_excel(writer, index = False, sheet_name = 'Tracts')
            # df_tracts2.to_excel(writer, index = False, sheet_name = 'Tracts wide')
    
    if geography == 'Counties':
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
        with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
            # df_counties2.to_excel(writer, index = False, sheet_name = 'Counties wide')
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
        with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_mpo1.to_excel(writer, index = False, sheet_name = 'MPO')
            # df_mpo2.to_excel(writer, index = False, sheet_name = 'MPO wide')
    
    if geography == 'MSA':
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
        with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_msa1.to_excel(writer, index = False, sheet_name = 'MSA')
            # df_msa2.to_excel(writer, index = False, sheet_name = 'MSA wide')


if sample_type == 'PUMS':
    if geography == 'PUMA':
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' PUMA '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
        #     df_puma.to_excel(writer, index = False, sheet_name = 'PUMA')
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
        with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_counties.to_excel(writer, index = False, sheet_name = 'Counties')
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
        #     df_msa.to_excel(writer, index = False, sheet_name = 'MSA')
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
        with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_mpo.to_excel(writer, index = False, sheet_name = 'MPO')
            
if sample_type == 'FOODSEC':
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
        with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_counties.to_excel(writer, index = False, sheet_name = 'Counties')
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
        with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_mpo.to_excel(writer, index = False, sheet_name = 'MPO')

print('')
print("Successfully exported")